In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from config import set_environment

# for the keys - as explained early in chapter 2
set_environment()

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Structured output

Let's define the data structure that describes a plan to solve a complex task:

In [3]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate


class Step(BaseModel):
    """A step that is a part of the plan to solve the task."""
    step: str = Field(description="Description of the step")


class Plan(BaseModel):
    """A plan to solve the task."""
    steps: list[Step]


prompt = PromptTemplate.from_template(
    "Prepare a step-by-step plan to solve the give task.\'n"
    "TASK:\n{task}\n"
)

We can explore that the model has succcessfully generated a complex _Pydantic_ structure, and all we needed to do was using a `with_structured_output` method:

In [6]:
result = (prompt | llm.with_structured_output(Plan)).invoke("How to shoot a basketball with good form?")

assert isinstance(result, Plan)
print(f"Amount of steps: {len(result.steps)}")

for idx, step in enumerate(result.steps):
    print(f"STEP {idx+1}: {step.step}\n")

Amount of steps: 8
STEP 1: Find a comfortable shooting stance with your feet shoulder-width apart, knees slightly bent, and your dominant foot slightly forward.

STEP 2: Grip the basketball with your shooting hand's fingertips, creating a small gap between your palm and the ball. Your guide hand (non-shooting hand) should be on the side of the ball for support.

STEP 3: Bring the ball up to your 'shooting pocket,' which is typically around your chest or shoulder height, keeping your elbow directly under the ball and pointing towards the rim.

STEP 4: Keep your eyes focused on the rim throughout the entire shot.

STEP 5: Initiate the shot by extending your knees and hips, transferring power upwards into your shooting arm.

STEP 6: As you extend your shooting arm, release the ball at the peak of your extension, using a strong wrist flick (snapping your wrist downwards).

STEP 7: Ensure your guide hand only supports the ball and does not interfere with the shooting motion; it should come 

We can also use a `json_mode` and pass a custom schema to an LLM. 

In [8]:
plan_schema = {
    "type": "ARRAY",
    "items": {
        "type": "OBJECT",
        "properties": {
            "step": {"type": "STRING"}
        },
    },
}

query = "How to shoot a basketball with good form?"
result = (prompt | llm.with_structured_output(schema=plan_schema, method="json_mode")).invoke(query)

In [9]:
assert(isinstance(result, list))
print(f"Amount of steps: {len(result)}")
print(result[0])

Amount of steps: 7
{'step': 'Find your balance: Stand with your feet shoulder-width apart, knees slightly bent, and your dominant foot slightly forward for stability.'}


As an alternative, we can use custom arguments supported by the LLM provider. Please, note that these options are vendor-specific, and you need to check the corresponding documentatino for supported extra arguments and formats:

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

llm_json = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    response_mime_type="application/json",
    response_schema=plan_schema,
)
result = (prompt | llm_json | JsonOutputParser()).invoke(
    f"{query}",
    # refer to https://reference.langchain.com/python/integrations/langchain_google_genai/ChatGoogleGenerativeAI/#langchain_google_genai.ChatGoogleGenerativeAI.convert_system_message_to_human
    # "The model also needs to be prompted to output the appropriate response type, otherwise the behavior is undefined."
    response_json_schema=plan_schema,
)
# result = (prompt | llm_json | JsonOutputParser()).invoke(query)
assert isinstance(result, list)
# print(f"Amount of steps: {len(result)}")
# print(result[0])
print(result)

OutputParserException: Invalid json output: The tone of the customer's "review" is:

*   **Inquisitive / Questioning:** The customer is clearly asking a question to gain knowledge or understanding.
*   **Seeking Information / Seeking Guidance:** The primary purpose is to get instructions or advice on a specific topic.
*   **Neutral:** There are no strong positive or negative emotions expressed. It's a straightforward request for information.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

We can also generated a enum (again, it's a vendor-dependent feature):

In [ ]:
from langchain_core.output_parsers import StrOutputParser

response_schema = {"type": "STRING", "enum": ["positive", "negative", "neutral"]}

prompt = PromptTemplate.from_template(
    "Classify the tone of the following customer's review"
    "\n{review}\n"
)

review = "I Like this movie!"
llm_enum = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    response_mime_type="text/x.enum",
    response_schema=response_schema,
)
result = (prompt | llm_enum | StrOutputParser()).invoke(review)
print(result)

The tone is **positive** and **enthusiastic**.

It's also very **simple** and **direct**. The exclamation mark emphasizes the strong approval and excitement.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

response_schema = {"type": "STRING", "enum": ["positive", "negative", "neutral"]}

prompt = PromptTemplate.from_template(
    "Classify the tone of the following customer's review:"
    "\n{review}\n"
)

review = "I like this movie!"
llm_enum = ChatVertexAI(model_name="gemini-2.0-flash", response_mime_type="text/x.enum", response_schema=response_schema)
result = (prompt | llm_enum | StrOutputParser()).invoke(review)
print(result)

positive
